# Base pricing env

> Base environment with some basic funcitons

In [ ]:
#| default_exp envs.pricing.base

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from abc import ABC, abstractmethod
from typing import Union, Tuple, List

from ddopai.envs.base import BaseEnvironment
from ddopai.utils import Parameter, MDPInfo
from ddopai.dataloaders.base import BaseDataLoader
from ddopai.loss_functions import pinball_loss

import gymnasium as gym

import numpy as np
import time

In [ ]:
#| export
class BasePricingEnv(BaseEnvironment):
    """
    Base class for inventory management environments. This class inherits from BaseEnvironment.
    
    """

    def __init__(self, 

        ## Parameters for Base env:
        mdp_info: MDPInfo, #
        postprocessors: list[object] | None = None,  # default is empty list
        mode: str = "train", # additional mode for the pricing environment TODO: add online mode to training loop
        return_truncation: str = True, # whether to return a truncated condition in step function
        dataloader: BaseDataLoader = None, # dataloader for the environment
        horizon_train: int = 100
        ) -> None:

        self.dataloader = dataloader
        self.episode = 0
        
        super().__init__(mdp_info=mdp_info, postprocessors = postprocessors,  mode = mode, return_truncation=return_truncation, horizon_train=horizon_train)
    
    def set_observation_space(self,
                            shape: tuple, # shape of the dataloader features
                            low: Union[np.ndarray, float] = -np.inf, # lower bound of the observation space
                            high: Union[np.ndarray, float] = np.inf, # upper bound of the observation space
                            samples_dim_included = True # whether the first dimension of the shape input is the number of samples
                            ) -> None:
        
        '''
        Set the observation space of the environment.
        This is a standard function for simple observation spaces. For more complex observation spaces,
        this function should be overwritten. Note that it is assumped that the first dimension
        is n_samples that is not relevant for the observation space.

        '''

        # To handle cases when no external information is available (e.g., parametric NV)
        
        if shape is None:
            self.observation_space = None

        else:
            if not isinstance(shape, tuple):
                raise ValueError("Shape must be a tuple.")
            
            if samples_dim_included:
                shape = shape[1:] # assumed that the first dimension is the number of samples

            self.observation_space = gym.spaces.Box(low=low, high=high, shape=shape, dtype=np.float32)

    def set_action_space(self,
                            shape: tuple, # shape of the dataloader target
                            low: Union[np.ndarray, float] = -np.inf, # lower bound of the observation space
                            high: Union[np.ndarray, float] = np.inf, # upper bound of the observation space
                            samples_dim_included = True # whether the first dimension of the shape input is the number of samples
                            ) -> None:
        
        '''
        Set the action space of the environment.
        This is a standard function for simple action spaces. For more complex action spaces,
        this function should be overwritten. Note that it is assumped that the first dimension
        is n_samples that is not relevant for the action space.
        '''

        if not isinstance(shape, tuple):
            raise ValueError("Shape must be a tuple.")
        
        if samples_dim_included:
            shape = shape[1:] # assumed that the first dimension is the number of samples

        self.action_space = gym.spaces.Box(low=low, high=high, shape=shape, dtype=np.float32)
    
    def get_observation(self):
        
        """
        Return the current observation. This function is for the online learning case it will return only the state,
        this function should be overwritten.

        """

        X_item, Y_item  = self.dataloader[self.index]
        self.X_item = X_item
        self.Y_item = Y_item
        return X_item, Y_item
    
    def reset(self,
        start_index: int | str = None, # index to start from
        state: np.ndarray = None # initial state
        ) -> Tuple[np.ndarray, bool]:

        """
        Reset function for the Newsvendor problem. It will return the first observation and demand.
        For val and test modes, it will by default reset to 0, while for the train mode it depends
        on the paramter "horizon_train" whether a random point in the training data is selected or 0
        """
        start_index = self.reset_index_from_episode(start_index) # reset the index from the episode
        truncated = self.reset_index(start_index)
        
            
        observation, demand = self.get_observation()
        return observation
    
    def reset_index_from_episode(self,start_index: int | str = None) -> int:
        
        if start_index == "random":
            if self.mode == "train":
                self.episode = np.random.choice(range(0, len(self.train_tasks)))
            else:
                raise ValueError("start_index cannot be 'random' in val or test mode")
            
        elif isinstance(start_index, int):
            self.episode = start_index
        else:
            self.episode = 0
        start_index = int(self.episode * self.mdp_info.horizon)
        
        if self.mode == "train":
            self.task = self.train_tasks[self.episode]
        elif self.mode == "val":
            self.task = self.val_tasks[self.episode]
        elif self.mode == "test":
            self.task = self.test_tasks[self.episode]
        return start_index
        
    def reset_index(self,
    start_index: Union[int,str], 
    ) -> bool:

        """

        Reset the index of the environment. If start_index is an integer, the index is set to this value. If start_index is "random",
        the index is set to a random integer between 0 and the length of the training data.

        """

        start_index = self.get_start_index(start_index) # Returns the start index or 0 for None values

        if start_index=="random":
            if self.mode == "train":
                if self.dataloader.len_train is not None and self.dataloader.len_train > self.mdp_info.horizon:
                    random_index = np.random.choice(range(0, self.dataloader.len_train-self.mdp_info.horizon, self.mdp_info.horizon))
                else:
                    random_index = 0
                self.start_index = random_index 
            else:
                raise ValueError("start_index cannot be 'random' in val or test mode")
        elif isinstance(start_index, int):
            self.start_index = start_index
        else:
            raise ValueError("start_index must be an integer or 'random'")
        
        if self.dataloader.len_train is not None:
            self.max_index = self.dataloader.len_train if self.mode == "train" else self.dataloader.len_val if self.mode == "val" else self.dataloader.len_test
            self.max_index -= 1
        else:
            self.max_index = self.start_index+self.mdp_info.horizon
        self.max_index_episode = np.minimum(self.max_index, self.start_index+self.mdp_info.horizon)
        if self.mode == "test" or self.mode == "val":
            self.max_index_episode += 1

        truncated = self.set_index(self.start_index) # assuming we only start randomly during training.
        
        return truncated

    def train(self, update_mdp_info=True):
        """
        Set the environment in training mode by both setting the internal state self._train and the dataloader. 
        If the horizon is set to "use_all_data", the horizon is set to the length of the training data, otherwise
        it is set to the horizon_train attribute of the environment. Finally, the function updates the MDP info
        and resets with the new state.

        """
        self._mode = "train"

        if hasattr(self, "dataloader"):
            self.dataloader.train()

            if hasattr(self, "horizon_train"):
                if self.horizon_train == "use_all_data":
                    horizon = self.dataloader.len_train
                else:
                    horizon = self.horizon_train
        else:
            horizon = self.mdp_info.horizon

        if update_mdp_info:
            self.update_mdp_info(gamma=self.mdp_info.gamma, horizon=horizon)

        self.reset()
    
    def val(self, update_mdp_info=True):
        """
        Set the environment in validation mode by both setting the internal state self._val and the dataloader.
        The horizon is kept like the horizon of the mdp. Finally, the function updates the MDP info
        and resets with the new state.

        """
        self._mode = "val"

        if hasattr(self, "dataloader"):
            self.dataloader.val()

        if update_mdp_info:
            self.update_mdp_info(gamma=self.mdp_info.gamma, horizon=self.mdp_info.horizon)

        self.reset()

    def test(self, update_mdp_info=True):
        """
        Set the environment in testing mode by both setting the internal state self._test and the dataloader.
        The horizon of test is always set to the length of the test data. Finally, the function updates the MDP info
        and resets with the new state.

        """
        self._mode = "test"

        if hasattr(self, "dataloader"):
            self.dataloader.test()
            

        if update_mdp_info:
            self.update_mdp_info(gamma=self.mdp_info.gamma, horizon=self.mdp_info.horizon)

        self.reset()